# AlphaRank: Multi-Agent Evaluation by Evolution

This notebook implements AlphaRank, a ranking algorithm for multi-agent systems that handles non-transitive relationships (like Rock-Paper-Scissors) where Elo ratings fail.

## Background

**Elo Rating System:**
- Assumes transitive skill: if A > B and B > C, then A > C
- Works well for chess, Go, and many competitive games
- Fails when there are cyclic win patterns

**AlphaRank:**
- Based on evolutionary game theory (replicator dynamics)
- Models the population distribution over strategies
- Handles non-transitive games naturally
- Produces a stationary distribution indicating which strategies dominate

## References

- Omidshafiei, S., et al. (2019). "AlphaRank: Multi-Agent Evaluation by Evolution". arXiv:1903.01373

In [ ]:
import numpy as np
from numpy.linalg import eig
import matplotlib.pyplot as plt
from typing import Optional

# Add parent directory to path for imports
import sys
sys.path.insert(0, '..')

from ratings.payoff_matrix import PayoffMatrix, compute_payoff_matrix_from_csv

## AlphaRank Implementation

AlphaRank works by constructing a transition matrix from the payoff matrix and computing its stationary distribution (dominant eigenvector). The stationary distribution represents the equilibrium population frequencies.

In [ ]:
def compute_alpha_rank(
    payoff_matrix: np.ndarray,
    alpha: float = 1.0,
    temperature: float = 1.0,
) -> np.ndarray:
    """Compute AlphaRank stationary distribution from a payoff matrix.

    Parameters
    ----------
    payoff_matrix:
        Square matrix M where M[i,j] is the win rate of agent i vs agent j.
        Values should be in [0, 1].
    alpha:
        Selection intensity parameter. Higher values emphasize differences.
        Default 1.0 provides moderate selection pressure.
    temperature:
        Temperature for softmax normalization. Higher = more uniform.
        Default 1.0.

    Returns
    -------
    np.ndarray
        Stationary distribution (probability mass) over agents.
        Higher values indicate stronger agents.
    """
    n = payoff_matrix.shape[0]
    assert payoff_matrix.shape == (n, n), "Payoff matrix must be square"

    # Convert win rates to fitness using softmax-like transformation
    # This emphasizes strong performers and de-emphasizes weak ones
    exp_payoff = np.exp(alpha * payoff_matrix / temperature)

    # Build transition matrix P where P[i,j] is probability of transitioning
    # from strategy j to strategy i (column-stochastic)
    #
    # The key insight: an agent is more likely to be "copied" if it performs
    # well against the current population
    P = np.zeros((n, n))

    for j in range(n):
        # For each current strategy j, compute transition probabilities
        # based on how well each strategy i performs against j
        for i in range(n):
            if i != j:
                # Probability of switching from j to i based on i's payoff vs j
                P[i, j] = exp_payoff[i, j] - 1  # Positive if i beats j

    # Add self-transition probabilities (stay with current strategy)
    # Based on how well current strategy does against itself
    for j in range(n):
        P[j, j] = 1.0 + np.sum(exp_payoff[:, j]) - exp_payoff[j, j] - n

    # Ensure non-negative and normalize columns
    P = np.maximum(P, 0)
    col_sums = P.sum(axis=0)
    col_sums[col_sums == 0] = 1  # Avoid division by zero
    P = P / col_sums[np.newaxis, :]

    # Compute stationary distribution (dominant eigenvector of P)
    # The stationary distribution π satisfies: P @ π = π
    eigenvalues, eigenvectors = eig(P)

    # Find eigenvalue closest to 1 (should be the dominant one)
    idx = np.argmin(np.abs(eigenvalues - 1.0))
    stationary = np.real(eigenvectors[:, idx])

    # Normalize to probability distribution
    stationary = np.abs(stationary)
    stationary = stationary / stationary.sum()

    return stationary


def compute_alpha_rank_simple(
    payoff_matrix: np.ndarray,
    m: float = 10.0,
) -> np.ndarray:
    """Simplified AlphaRank using a replicator dynamics approach.

    This is an alternative implementation that may be more stable
    for small payoff matrices.

    Parameters
    ----------
    payoff_matrix:
        Square matrix M where M[i,j] is the win rate of agent i vs agent j.
    m:
        Population size parameter (larger = more stable equilibrium).

    Returns
    -------
    np.ndarray
        Stationary distribution over agents.
    """
    n = payoff_matrix.shape[0]

    # Initialize uniform distribution
    p = np.ones(n) / n

    # Compute fitness for each strategy against the population
    # f_i = sum_j M[i,j] * p_j (expected payoff)
    fitness = payoff_matrix @ p

    # Average fitness
    avg_fitness = p @ fitness

    # Build transition matrix for Markov chain
    # P[i,j] = probability of strategy j being replaced by strategy i
    P = np.zeros((n, n))

    for j in range(n):
        for i in range(n):
            if i != j:
                # Transition probability based on relative fitness
                fitness_diff = fitness[i] - fitness[j]
                if fitness_diff > 0:
                    P[i, j] = (1 / m) * (fitness_diff / (np.max(fitness) - np.min(fitness) + 1e-10))

    # Self-transitions
    for j in range(n):
        P[j, j] = 1.0 - P[:, j].sum()

    # Ensure valid probability matrix
    P = np.maximum(P, 0)
    P = P / P.sum(axis=0, keepdims=True)

    # Power iteration to find stationary distribution
    for _ in range(1000):
        p_new = P @ p
        if np.allclose(p, p_new, atol=1e-10):
            break
        p = p_new

    return p / p.sum()

## Synthetic Example: Rock-Paper-Scissors

Let's start with the classic non-transitive game where Elo would fail.

In [ ]:
# Rock-Paper-Scissors payoff matrix
# Row agent vs column agent: 1 = win, 0 = loss, 0.5 = tie
rps_agents = ["Rock", "Paper", "Scissors"]
rps_matrix = np.array([
    # vs: Rock  Paper  Scissors
    [0.5,  0.0,  1.0],    # Rock
    [1.0,  0.5,  0.0],    # Paper
    [0.0,  1.0,  0.5],    # Scissors
])

print("Rock-Paper-Scissors Payoff Matrix:")
print("Rows: agent playing, Columns: opponent")
print()
print(f"{'':>10}", end="")
for agent in rps_agents:
    print(f"{agent:>10}", end="")
print()
for i, agent in enumerate(rps_agents):
    print(f"{agent:>10}", end="")
    for j in range(len(rps_agents)):
        print(f"{rps_matrix[i,j]:>10.1f}", end="")
    print()

In [ ]:
# Compute AlphaRank for RPS
rps_ranking = compute_alpha_rank(rps_matrix, alpha=1.0)

print("\nAlphaRank Results for Rock-Paper-Scissors:")
print("=" * 50)
for agent, score in zip(rps_agents, rps_ranking):
    print(f"{agent:>15}: {score:.4f} ({score*100:.1f}%)")

print(f"\n{'='*50}")
print("Note: All three strategies have equal weight (~33%)")
print("This is correct! In RPS, no strategy dominates.")

## Synthetic Example: Imbalanced Game

Now let's try a game where one strategy has a slight advantage.

In [ ]:
# Imbalanced game: A beats B most of the time, B and C are even
imbalanced_agents = ["Agent A (Strong)", "Agent B (Weak)", "Agent C (Medium)"]
imbalanced_matrix = np.array([
    # vs:  A     B     C
    [0.5,  0.9,  0.7],   # A
    [0.1,  0.5,  0.5],   # B
    [0.3,  0.5,  0.5],   # C
])

print("Imbalanced Game Payoff Matrix:")
print("Rows: agent playing, Columns: opponent")
print()
print(f"{'':>20}", end="")
for agent in ["A", "B", "C"]:
    print(f"{agent:>10}", end="")
print()
for i, agent in enumerate(["A", "B", "C"]):
    print(f"{agent:>20}", end="")
    for j in range(3):
        print(f"{imbalanced_matrix[i,j]:>10.1f}", end="")
    print()

In [ ]:
# Compute AlphaRank for imbalanced game
imbalanced_ranking = compute_alpha_rank(imbalanced_matrix, alpha=2.0)

print("\nAlphaRank Results for Imbalanced Game:")
print("=" * 50)
for agent, score in zip(imbalanced_agents, imbalanced_ranking):
    print(f"{agent:>20}: {score:.4f} ({score*100:.1f}%)")

print(f"\n{'='*50}")
print("Agent A dominates due to high win rates against others.")

## Synthetic Example: Tournament Results

Let's simulate a more realistic tournament scenario with 5 agents.

In [ ]:
# 5-agent tournament with transitive + non-transitive elements
# MCTS-500 is strongest, but LLM beats it (non-transitivity)
tournament_agents = ["Random", "MCTS-50", "MCTS-200", "MCTS-500", "LLM"]

tournament_matrix = np.array([
    # vs:  Rand  M50   M200  M500  LLM
    [0.5,  0.1,  0.05, 0.02, 0.3],   # Random
    [0.9,  0.5,  0.3,  0.2,  0.4],   # MCTS-50
    [0.95, 0.7,  0.5,  0.35, 0.45],  # MCTS-200
    [0.98, 0.8,  0.65, 0.5,  0.4],   # MCTS-500
    [0.7,  0.6,  0.55, 0.6,  0.5],   # LLM (beats MCTS-500!)
])

print("Tournament Payoff Matrix (win rates):")
print("=" * 60)
print(f"{'':>12}", end="")
for agent in ["Rand", "M50", "M200", "M500", "LLM"]:
    print(f"{agent:>8}", end="")
print()
short_names = ["Rand", "M50", "M200", "M500", "LLM"]
for i, name in enumerate(short_names):
    print(f"{name:>12}", end="")
    for j in range(5):
        print(f"{tournament_matrix[i,j]:>8.2f}", end="")
    print()

In [ ]:
# Compare AlphaRank with average win rate (simple ranking)
alpha_ranking = compute_alpha_rank(tournament_matrix, alpha=2.0)
avg_win_rates = tournament_matrix.mean(axis=1)

# Sort by AlphaRank score
sorted_indices = np.argsort(alpha_ranking)[::-1]

print("\nTournament Rankings Comparison:")
print("=" * 70)
print(f"{'Rank':>6} {'Agent':>12} {'AlphaRank':>12} {'Avg Win Rate':>14}")
print("-" * 70)

for rank, idx in enumerate(sorted_indices, 1):
    print(f"{rank:>6} {tournament_agents[idx]:>12} "
          f"{alpha_ranking[idx]:>12.4f} {avg_win_rates[idx]:>14.2%}")

print("\n" + "=" * 70)
print("Note: LLM ranks higher in AlphaRank because it beats the")
print("strongest MCTS agent, creating a non-transitive advantage.")

## Visualizing AlphaRank Results

In [ ]:
def plot_ranking_comparison(
    agents: list[str],
    alpha_scores: np.ndarray,
    avg_win_rates: np.ndarray,
    title: str = "AlphaRank vs Average Win Rate",
):
    """Plot AlphaRank scores alongside average win rates."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Sort by AlphaRank
    sorted_idx = np.argsort(alpha_scores)[::-1]
    sorted_agents = [agents[i] for i in sorted_idx]
    sorted_alpha = alpha_scores[sorted_idx]
    sorted_winrate = avg_win_rates[sorted_idx]

    x = np.arange(len(agents))

    # AlphaRank bar chart
    bars1 = axes[0].bar(x, sorted_alpha, color='steelblue', edgecolor='black')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(sorted_agents, rotation=45, ha='right')
    axes[0].set_ylabel('AlphaRank Score')
    axes[0].set_title('AlphaRank Distribution')
    axes[0].set_ylim(0, max(sorted_alpha) * 1.2)

    # Add value labels
    for bar, val in zip(bars1, sorted_alpha):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=9)

    # Average win rate bar chart
    bars2 = axes[1].bar(x, sorted_winrate, color='coral', edgecolor='black')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(sorted_agents, rotation=45, ha='right')
    axes[1].set_ylabel('Average Win Rate')
    axes[1].set_title('Average Win Rate')
    axes[1].set_ylim(0, 1.0)

    # Add value labels
    for bar, val in zip(bars2, sorted_winrate):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{val:.1%}', ha='center', va='bottom', fontsize=9)

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Plot the tournament results
plot_ranking_comparison(
    tournament_agents,
    alpha_ranking,
    avg_win_rates,
    "Tournament: AlphaRank vs Average Win Rate"
)

In [ ]:
def plot_payoff_heatmap(
    matrix: np.ndarray,
    agents: list[str],
    title: str = "Payoff Matrix",
):
    """Plot payoff matrix as a heatmap."""
    fig, ax = plt.subplots(figsize=(8, 6))

    im = ax.imshow(matrix, cmap='RdYlGn', vmin=0, vmax=1)

    # Add colorbar
    cbar = ax.figure.colorbar(im, ax=ax)
    cbar.ax.set_ylabel('Win Rate', rotation=-90, va="bottom", fontsize=12)

    # Set ticks and labels
    ax.set_xticks(np.arange(len(agents)))
    ax.set_yticks(np.arange(len(agents)))
    ax.set_xticklabels(agents, rotation=45, ha='right')
    ax.set_yticklabels(agents)

    # Add text annotations
    for i in range(len(agents)):
        for j in range(len(agents)):
            text = ax.text(j, i, f'{matrix[i, j]:.2f}',
                          ha='center', va='center', color='black', fontsize=10)

    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Opponent', fontsize=12)
    ax.set_ylabel('Agent', fontsize=12)

    plt.tight_layout()
    plt.show()

# Plot the tournament payoff matrix
plot_payoff_heatmap(tournament_matrix, tournament_agents, "Tournament Payoff Matrix")

## Loading Real Tournament Data

To use AlphaRank with real tournament results, load a CSV file using the payoff matrix module.

In [ ]:
# Example: Load from a real CSV file (if available)
# Uncomment and modify the path to use your own data

# csv_path = '../results/your_results_file.csv'
# matrix = compute_payoff_matrix_from_csv(csv_path)
# payoff_np = matrix.to_numpy()
# ranking = compute_alpha_rank(payoff_np, alpha=2.0)

# print(f"AlphaRank Results for {matrix.game}:")
# for agent, score in sorted(zip(matrix.agents, ranking), key=lambda x: -x[1]):
#     print(f"  {agent}: {score:.4f}")

## Summary

### When to Use AlphaRank vs Elo

| Criterion | Elo | AlphaRank |
|-----------|-----|----------|
| **Transitive games** | ✅ Best | ✅ Works |
| **Non-transitive games (RPS)** | ❌ Fails | ✅ Best |
| **Simple interpretation** | ✅ Easy | ⚠️ Moderate |
| **Computational cost** | ✅ O(n) | ⚠️ O(n²) |
| **Single rating number** | ✅ Yes | ❌ Distribution |
| **Handles cycles** | ❌ No | ✅ Yes |

### Key Takeaways

1. **Elo is great for transitive games** where skill hierarchies are clear (chess, tennis).

2. **AlphaRank handles non-transitivity** where A beats B, B beats C, but C beats A.

3. **AlphaRank produces a population distribution** rather than individual scores, which captures equilibrium strategies in metagames.

4. **Use both** in practice: Elo for quick rankings, AlphaRank when you suspect non-transitive dynamics.